# 01 — Extract Shot Attempts
Load Wyscout event data league-by-league, filter to shot attempts (standard shots,
in-game penalties, and direct free-kick shots), extract coordinates and goal outcomes,
then save combined, train, and test parquet files.

> **"Shot attempts"** in this notebook means: `Shot/Shot`, `Free Kick/Penalty` (matchPeriod ≠ P),
> and `Free Kick/Free kick shot`. Legacy filenames retain "shots" for continuity.

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import pandas as pd
from IPython.display import display
from src.config import INTERIM_DIR as DATA_DIR, EVENT_DIR, ALL_LEAGUES, TRAIN_LEAGUES, TEST_LEAGUES
from src.data_loader import load_events
from src.shot_filtering import extract_shot_attempts, save_shot_splits, KEEP_COLS

## Step 0: Path validation

In [2]:
assert EVENT_DIR.exists(), f"Missing directory: {EVENT_DIR}"
assert (EVENT_DIR / "events_England.json").exists(), "England events file not found"
print("Paths OK")

Paths OK


## Step 1: Helper functions

## Step 2: Load shot attempts per league (memory-efficient)
Filter to shot attempts at read time — this includes:
- `Shot / Shot` — standard open-play shots
- `Free Kick / Penalty` — in-game awarded penalties (matchPeriod ≠ "P")
- `Free Kick / Free kick shot` — direct free-kick shots

Penalty shootout kicks (matchPeriod == "P") are excluded — different context, would skew xG estimates.

In [ ]:
events_by_league = {}
for league in ALL_LEAGUES:
    events = load_events(league)
    league_shots = extract_shot_attempts(events, league)
    n_pen = league_shots["is_penalty"].sum()
    n_fk  = league_shots["is_direct_free_kick"].sum()
    print(f"{league}: {len(league_shots):,} attempts  (penalties: {n_pen}, direct FK shots: {n_fk})")
    events_by_league[league] = league_shots

train_df = pd.concat([events_by_league[lg] for lg in TRAIN_LEAGUES], ignore_index=True)
test_df  = pd.concat([events_by_league[lg] for lg in TEST_LEAGUES],  ignore_index=True)

print(f"\nTrain shots: {len(train_df):,}")
print(f"Test shots:  {len(test_df):,}")

## Step 3: Build train/test DataFrames
Training leagues: France, Germany, Italy, Spain  
Test league: England (held out)

## Step 4: Combine and preview

In [4]:
# Cast dtypes to match pre-refactor parquet schema.
# pd.to_numeric handles any non-numeric x/y gracefully; assert no nulls before int cast.
for df in [train_df, test_df]:
    df["x"] = pd.to_numeric(df["x"], errors="coerce")
    df["y"] = pd.to_numeric(df["y"], errors="coerce")
    assert df["x"].isna().sum() == 0, "Unexpected NaN in x after coerce"
    assert df["y"].isna().sum() == 0, "Unexpected NaN in y after coerce"
    df["x"] = df["x"].astype("int64")
    df["y"] = df["y"].astype("int64")
    for col in ["is_goal", "is_penalty", "is_direct_free_kick"]:
        df[col] = df[col].astype("int8")

In [ ]:
shots = save_shot_splits(train_df, test_df, DATA_DIR)
print(f"Saved {len(shots):,} combined attempts  → wyscout_shots.parquet")
print(f"Saved {len(train_df):,} train attempts   → wyscout_train_shots.parquet")
print(f"Saved {len(test_df):,} test attempts    → wyscout_test_shots.parquet")
print("Saved 100-row sample                   → wyscout_shots_sample.csv")
print(f"\nColumns: {list(shots.columns)}")

In [6]:
shots[["league", "subEventName", "x", "y", "is_goal"]].head()

,league,subEventName,x,y,is_goal
0,France,Shot,94,57,1
1,France,Shot,83,42,0
2,France,Shot,96,43,1
3,France,Shot,84,21,0
4,France,Shot,73,51,0


## Step 5: Sanity checks

> **Note:** This notebook extracts shot-level event data only.  
> Match dates and chronological ordering will be added in the next notebook using the matches files.

In [7]:
print("Missing coordinates:")
print(shots[["x", "y"]].isna().sum())

Missing coordinates:
x    0
y    0
dtype: int64


In [8]:
print("Coordinate ranges:")
print(f"x: {shots['x'].min()} to {shots['x'].max()}")
print(f"y: {shots['y'].min()} to {shots['y'].max()}")

Coordinate ranges:
x: 1 to 100
y: 0 to 100


In [9]:
print(shots[["x", "y", "is_goal"]].dtypes)

x          int64
y          int64
is_goal     int8
dtype: object


In [ ]:
ALLOWED_PAIRS = {
    ("Shot",      "Shot"),
    ("Free Kick", "Penalty"),
    ("Free Kick", "Free kick shot"),
}
actual_pairs = set(zip(shots["eventName"], shots["subEventName"]))
assert actual_pairs == ALLOWED_PAIRS, (
    f"Extra pairs: {actual_pairs - ALLOWED_PAIRS} | Missing pairs: {ALLOWED_PAIRS - actual_pairs}"
)

assert (shots["is_penalty"] == 1).sum() > 0, "No penalties found — check extraction filter"
assert (shots["is_direct_free_kick"] == 1).sum() > 0, "No direct FK shots found — check extraction filter"

# Mutual exclusivity
assert not ((shots["is_penalty"] == 1) & (shots["is_direct_free_kick"] == 1)).any(), \
    "A row has both is_penalty and is_direct_free_kick set to 1"

# Penalty shootout guard
assert not ((shots["is_penalty"] == 1) & (shots["matchPeriod"] == "P")).any(), \
    "Penalty shootout kick found — check matchPeriod filter"

print("Event-type counts:")
print(shots.groupby(["eventName", "subEventName"]).size().to_string())
print(f"\nTotal attempts: {len(shots):,}")
print(f"  Standard shots:   {((shots['is_penalty'] == 0) & (shots['is_direct_free_kick'] == 0)).sum():,}")
print(f"  Penalties:        {(shots['is_penalty'] == 1).sum():,}")
print(f"  Direct FK shots:  {(shots['is_direct_free_kick'] == 1).sum():,}")

In [11]:
print("Duplicate rows:", shots.duplicated().sum())

Duplicate rows: 0


In [12]:
print(f"Total attempts: {len(shots):,}")
print(f"Goals:          {shots['is_goal'].sum():,}")
print(f"Goal rate:      {shots['is_goal'].mean():.2%}")

league_summary = shots.groupby("league").agg(
    attempts=("is_goal", "count"),
    penalties=("is_penalty", "sum"),
    direct_fk_shots=("is_direct_free_kick", "sum"),
    goals=("is_goal", "sum"),
    goal_rate=("is_goal", "mean"),
)
print("\nLeague-level summary:")
display(league_summary)

Total attempts: 43,040
Goals:          4,790
Goal rate:      11.13%

League-level summary:


,attempts,penalties,direct_fk_shots,goals,goal_rate
league,,,,,
England,8881,80,350,988,0.111249
France,8977,129,521,998,0.111173
Germany,7290,93,299,833,0.114266
Italy,9347,126,415,978,0.104633
Spain,8545,113,453,993,0.116208


## Step 6: Save outputs

In [13]:
check_df = pd.read_parquet(DATA_DIR / "wyscout_shots.parquet")
assert check_df.shape[0] == len(shots), "Row count mismatch on reload"
expected_cols = set(KEEP_COLS)
assert expected_cols.issubset(check_df.columns), f"Missing columns: {expected_cols - set(check_df.columns)}"
assert "positions" not in check_df.columns, "positions column should not be in saved parquet"
assert "tags" not in check_df.columns, "tags column should not be in saved parquet"
print(check_df.shape)
check_df.head()

(43040, 13)


,league,matchId,playerId,teamId,eventSec,matchPeriod,eventName,subEventName,x,y,is_goal,is_penalty,is_direct_free_kick
0,France,2500686,256992,3799,605.975493,1H,Shot,Shot,94,57,1,0,0
1,France,2500686,334552,3772,859.236394,1H,Shot,Shot,83,42,0,0,0
2,France,2500686,26389,3772,1568.104834,1H,Shot,Shot,96,43,1,0,0
3,France,2500686,276920,3772,1800.852078,1H,Shot,Shot,84,21,0,0,0
4,France,2500686,366760,3799,2009.537139,1H,Shot,Shot,73,51,0,0,0
